 # Using SEC EDGAR RESTful data APIs

 This notebook shows how to retrieve information reported by regulated entities to U.S. Securities and Exchange Commision (SEC).

 SEC is maintainig EDGAR system with information about all regulated enties (companies, funds, individuals). Accessing the data is free and there is number of [various ways how to access the data](https://www.sec.gov/os/accessing-edgar-data).

 "data.sec.gov" was created to host RESTful data Application Programming Interfaces (APIs) delivering JSON-formatted data to external customers and to web pages on SEC.gov. These APIs do not require any authentication or API keys to access.

 Currently included in the APIs are the submissions history by filer and the XBRL data from financial statements (forms 10-Q, 10-K,8-K, 20-F, 40-F, 6-K, and their variants).

 The JSON structures are updated throughout the day, in real time, as submissions are disseminated.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# data are published in JSON format so we will need json library
import json

 # Finding CIK of company

 EDGAR assigns to filers a unique numerical identifier, known as a Central Index Key (CIK), when they sign up to make filings to the SEC. CIK numbers remain unique to the filer; they are not recycled.

 List of all CIKs matched with entity name is available for download [(13 MB, text file)](https://www.sec.gov/Archives/edgar/cik-lookup-data.txt). Note that this list includes funds and individuals and is historically cumulative for company names. Thus a given CIK may be associated with multiple names in the case of company or fund name changes, and the list contains some entities that no longer file with the SEC.

 We will be using smaller (611 kB) JSON [kaggle dataset](https://www.kaggle.com/datasets/svendaj/sec-edgar-cik-ticker-exchange), which is sourcing data directly at EDGAR and is input for this notebook. This dataset contains only companies names, CIK, ticker and associated stock exchange.

In [ ]:
# # Let's convert CIK JSON to pandas DataFrame
# # First load the data into python dictionary
# https://www.sec.gov/files/company_tickers_exchange.json
# with open("../input/sec-edgar-cik-ticker-exchange/company_tickers_exchange.json", "r") as f:
#     CIK_dict = json.load(f)

import requests

url = "https://www.sec.gov/files/company_tickers_exchange.json"

# The SEC requires you to identify yourself in the User-Agent header
headers = {
    "User-Agent": "Your Name YourEmail@example.com" 
}

# Fetch the data from the URL
response = requests.get(url, headers=headers)

# Check if the request was successful (Status code 200)
if response.status_code == 200:
    # .json() automatically parses the JSON text into a Python dictionary
    CIK_dict = response.json() 
    print("Successfully fetched the data!")
else:
    print(f"Failed to fetch data. Status code: {response.status_code}")

Successfully fetched the data!


In [ ]:
# dataset contains two sections
CIK_dict.keys()

dict_keys(['fields', 'data'])

In [ ]:
# fields is specifying meaning and and order of company data
# we will use it as columns names
CIK_dict["fields"]

['cik', 'name', 'ticker', 'exchange']

In [ ]:
# data section is list of records/lists for each company
# we will use it as DataFrame rows
print("Number of company records:", len(CIK_dict["data"]))
CIK_dict["data"][:5]    # first 5 records

Number of company records: 10438


[[1045810, 'NVIDIA CORP', 'NVDA', 'Nasdaq'],
 [1652044, 'Alphabet Inc.', 'GOOGL', 'Nasdaq'],
 [320193, 'Apple Inc.', 'AAPL', 'Nasdaq'],
 [789019, 'MICROSOFT CORP', 'MSFT', 'Nasdaq'],
 [1018724, 'AMAZON COM INC', 'AMZN', 'Nasdaq']]

In [ ]:
# convert CIK_dict to pandas
CIK_df = pd.DataFrame(CIK_dict["data"], columns=CIK_dict["fields"])
CIK_df

,cik,name,ticker,exchange
0,1045810,NVIDIA CORP,NVDA,Nasdaq
1,1652044,Alphabet Inc.,GOOGL,Nasdaq
2,320193,Apple Inc.,AAPL,Nasdaq
3,789019,MICROSOFT CORP,MSFT,Nasdaq
4,1018724,AMAZON COM INC,AMZN,Nasdaq
...,...,...,...,...
10433,2108180,NOF Corporation/ADR,NOFCF,OTC
10434,2108185,Azbil Corporation/ADR,YMATF,OTC
10435,2109234,BELIMO Holding AG/ADR,BLHWF,OTC
10436,2109545,Knowledge Atlas Technology Joint Stock Co Limi...,KATJF,OTC


 ## Select the ticker of company used in this example

 Subsequent information retrieval will be using selected `ticker` and associated CIK

In [ ]:
# finding company row with given ticker
ticker = "TSLA"
CIK_df[CIK_df["ticker"] == ticker]

,cik,name,ticker,exchange
7,1318605,"Tesla, Inc.",TSLA,Nasdaq


In [ ]:
CIK = CIK_df[CIK_df["ticker"] == ticker].cik.values[0]

In [ ]:
# finding companies containing substring in company name
substring = "oil"
CIK_df[CIK_df["name"].str.contains(substring, case=False)]

,cik,name,ticker,exchange
257,49938,IMPERIAL OIL LTD,IMO,NYSE
658,2071881,Tourmaline Oil Corp/ADR,TRMOY,OTC
752,1327068,"United States Oil Fund, LP",USO,NYSE
1262,1698990,Magnolia Oil & Gas Corp,MGY,NYSE
1301,717423,MURPHY OIL CORP,MUR,NYSE
1757,1104485,"NORTHERN OIL & GAS, INC.",NOG,NYSE
2765,74046,Oil-Dri Corp of America,ODC,NYSE
2886,1121484,"OIL STATES INTERNATIONAL, INC",OIS,NYSE
2908,1868917,Saturn Oil & Gas Inc.,OILSF,OTC
3075,1383058,Invesco DB Oil Fund,DBO,NYSE


 # Entity’s current filing history

 Each entity’s current filing history is available at the following URL:

 * https://data.sec.gov/submissions/CIK##########.json

 Where the ########## is the entity’s 10-digit Central Index Key (CIK), including leading zeros.

 This JSON data structure contains metadata such as current name, former name, and stock exchanges and ticker symbols of publicly-traded companies. The object’s property path contains at least one year’s of filing or to 1,000 (whichever is more) of the most recent filings in a compact columnar data array. If the entity has additional filings, files will contain an array of additional JSON files and the date range for the filings each one contains.

In [ ]:
# preparation of input data, using ticker and CIK set earlier
url = f"https://data.sec.gov/submissions/CIK{str(CIK).zfill(10)}.json"
url

'https://data.sec.gov/submissions/CIK0001318605.json'

 # Reading from RESTful API

 EDGAR requires that HTTP requests will be identified with proper [UserAgent in header and comply with fair use policy (currently max. 10 requests per second)](https://www.sec.gov/os/accessing-edgar-data). At minimum you need to supply your own e-mail adress in User-Agent field (otherwise you will get 403/Forbiden error). If you will provide Host field, please be sure use data.sec.gov server and not www.sec.gov as mentioned in example (this would result in 404/Not Found error).

In [ ]:
# read response from REST API with `requests` library and format it as python dict

import requests
header = {
  "User-Agent": "your.email@email.com"#, # remaining fields are optional
#    "Accept-Encoding": "gzip, deflate",
#    "Host": "data.sec.gov"
}

company_filings = requests.get(url, headers=header).json()
company_filings.keys()

dict_keys(['cik', 'entityType', 'sic', 'sicDescription', 'ownerOrg', 'insiderTransactionForOwnerExists', 'insiderTransactionForIssuerExists', 'name', 'tickers', 'exchanges', 'ein', 'lei', 'description', 'website', 'investorWebsite', 'category', 'fiscalYearEnd', 'stateOfIncorporation', 'stateOfIncorporationDescription', 'addresses', 'phone', 'flags', 'formerNames', 'filings'])

In [ ]:
company_filings["addresses"]

{'mailing': {'street1': '1 TESLA ROAD',
  'street2': None,
  'city': 'AUSTIN',
  'stateOrCountry': 'TX',
  'zipCode': '78725',
  'stateOrCountryDescription': 'TX',
  'isForeignLocation': 0,
  'foreignStateTerritory': None,
  'country': None,
  'countryCode': None},
 'business': {'street1': '1 TESLA ROAD',
  'street2': None,
  'city': 'AUSTIN',
  'stateOrCountry': 'TX',
  'zipCode': '78725',
  'stateOrCountryDescription': 'TX',
  'isForeignLocation': None,
  'foreignStateTerritory': None,
  'country': None,
  'countryCode': None}}

In [ ]:
company_filings["filings"]["recent"].keys()

dict_keys(['accessionNumber', 'filingDate', 'reportDate', 'acceptanceDateTime', 'act', 'form', 'fileNumber', 'filmNumber', 'items', 'core_type', 'size', 'isXBRL', 'isInlineXBRL', 'primaryDocument', 'primaryDocDescription'])

 # Creating DataFrame with submitted filings

 `company_filings["filings"]["recent"]` contains up to 1000 last submitted filings sorted from latest to oldest.

In [ ]:
company_filings_df = pd.DataFrame(company_filings["filings"]["recent"])
company_filings_df

,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,primaryDocument,primaryDocDescription
0,0000102909-26-002479,2026-03-27,,2026-03-27T17:36:06.000Z,34,SCHEDULE 13G/A,005-85943,26804894,,SCHEDULE 13G/A,7103,0,0,xslSCHEDULE_13G_X02/primary_doc.xml,
1,0001104659-26-025379,2026-03-09,2026-03-05,2026-03-09T23:00:14.000Z,,4,,,,4,10350,0,0,xslF345X05/tm268346-1_4seq1.xml,OWNERSHIP DOCUMENT
2,0001950047-26-002335,2026-03-06,,2026-03-06T22:42:03.000Z,33,144,001-34756,26732483,,144,5307,0,0,xsl144X01/primary_doc.xml,
3,0001104659-26-021746,2026-02-27,2026-02-25,2026-02-28T00:00:21.000Z,,4,,,,4,21615,0,0,xslF345X05/tm267481-1_4seq1.xml,OWNERSHIP DOCUMENT
4,0001950047-26-001763,2026-02-25,,2026-02-25T21:27:12.000Z,33,144,001-34756,26679106,,144,5350,0,0,xsl144X01/primary_doc.xml,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999,0001494732-18-000002,2018-02-14,2018-02-12,2018-02-15T02:37:33.000Z,,4,,,,4,7286,0,0,xslF345X03/edgardoc.xml,PRIMARY DOCUMENT
1000,0001494732-18-000001,2018-02-14,2018-02-12,2018-02-15T02:16:40.000Z,,4,,,,4,9083,0,0,xslF345X03/edgardoc.xml,PRIMARY DOCUMENT
1001,0001288257-18-000001,2018-02-14,2017-05-01,2018-02-15T01:57:34.000Z,,4/A,,,,4/A,10177,0,0,xslF345X03/primary_doc.xml,PRIMARY DOCUMENT
1002,0001193125-18-045807,2018-02-14,,2018-02-14T22:02:26.000Z,34,SC 13G/A,005-85943,18613282,,SC 13G/A,38495,0,0,d540304dsc13ga.htm,AMENDMENT NO. 7 TO SCHEDULE 13G


In [ ]:
# filter only Annual reports
company_filings_df[company_filings_df.form == "10-K"]

,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,primaryDocument,primaryDocDescription
5,0001628280-26-003952,2026-01-29,2025-12-31,2026-01-29T01:55:03.000Z,34,10-K,001-34756,26574326,,XBRL,15755490,1,1,tsla-20251231.htm,10-K
151,0001628280-25-003063,2025-01-30,2024-12-31,2025-01-30T01:42:33.000Z,34,10-K,001-34756,25570807,,XBRL,15788647,1,1,tsla-20241231.htm,10-K
265,0001628280-24-002390,2024-01-29,2023-12-31,2024-01-27T02:00:20.000Z,34,10-K,001-34756,24569853,,XBRL,15527801,1,1,tsla-20231231.htm,10-K
366,0000950170-23-001409,2023-01-31,2022-12-31,2023-01-31T02:29:15.000Z,34,10-K,001-34756,23570030,,XBRL,31445171,1,1,tsla-20221231.htm,10-K
466,0000950170-22-000796,2022-02-07,2021-12-31,2022-02-05T01:11:27.000Z,34,10-K,001-34756,22595227,,XBRL,29316024,1,1,tsla-20211231.htm,10-K
598,0001564590-21-004599,2021-02-08,2020-12-31,2021-02-08T12:27:23.000Z,34,10-K,001-34756,21598537,,XBRL,32860345,1,1,tsla-10k_20201231.htm,10-K
736,0001564590-20-004475,2020-02-13,2019-12-31,2020-02-13T12:12:18.000Z,34,10-K,001-34756,20606921,,XBRL,29961626,1,1,tsla-10k_20191231.htm,10-K
889,0001564590-19-003165,2019-02-19,2018-12-31,2019-02-19T11:10:16.000Z,34,10-K,001-34756,19613254,,10-K,30826751,1,0,tsla-10k_20181231.htm,10-K
996,0001564590-18-002956,2018-02-23,2017-12-31,2018-02-23T11:07:43.000Z,34,10-K,001-34756,18634585,,10-K,25498533,1,0,tsla-10k_20171231.htm,10-K


 # Accessing specific filing document

 Let's download latest Annual Report (10-K). Files are stored in browsable directory structure for CIK and accession-number:
 * https://www.sec.gov/Archives/edgar/data/{CIK}/{accession-number}/

In [ ]:
access_number = company_filings_df[company_filings_df.form == "10-K"].accessionNumber.values[0].replace("-", "")

file_name = company_filings_df[company_filings_df.form == "10-K"].primaryDocument.values[0]

url = f"https://www.sec.gov/Archives/edgar/data/{CIK}/{access_number}/{file_name}"
url

'https://www.sec.gov/Archives/edgar/data/1318605/000162828026003952/tsla-20251231.htm'

In [ ]:
# dowloading and saving requested document to working directory
req_content = requests.get(url, headers=header).content.decode("utf-8")

with open(file_name, "w") as f:
    f.write(req_content)

 ## and saving it as PDF


In [ ]:
# pip install weasyprint

In [ ]:
# import os
# import sys

# # Add Homebrew's lib directory to the fallback search path for WeasyPrint dependencies
# if sys.platform == 'darwin':
#     os.environ['DYLD_FALLBACK_LIBRARY_PATH'] = '/opt/homebrew/lib:' + os.environ.get('DYLD_FALLBACK_LIBRARY_PATH', '')

# from weasyprint import HTML

# HTML(string=req_content, base_url="").write_pdf(file_name + ".pdf")

In [ ]:
# !ls -al .

 # XBRL data APIs

 Extensible Business Markup Language (XBRL) is an XML-based format for reporting financial statements used by the SEC and financial regulatory agencies across the world. XBRL, in a separate XML file or more recently embedded in quarterly and annual HTML reports as inline XBRL, was first required by the SEC in 2009. XBRL facts must be associated for a standard US-GAAP or IFRS taxonomy. Companies can also extend standard taxonomies with their own custom taxonomies.

 The following XBRL APIs aggregate facts from across submissions that
 1. Use a non-custom taxonomy (e.g. us-gaap, ifrs-full, dei, or srt)
 1. Apply to the entire filing entity

 This ensures that facts have a consistent context and meaning across companies and between filings and are comparable between companies and across time.

 ## All company concepts data
 ## data.sec.gov/api/xbrl/companyfacts/

 This API returns all the company concepts data for a company into a single API call:

 * https://data.sec.gov/api/xbrl/companyfacts/CIK##########.json

In [ ]:
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{str(CIK).zfill(10)}.json"
url

'https://data.sec.gov/api/xbrl/companyfacts/CIK0001318605.json'

In [ ]:
company_facts = requests.get(url, headers=header).json()

# get the current assets values as reported over time and make it pandas DataFrame
curr_assets_df = pd.DataFrame(company_facts["facts"]["us-gaap"]["AssetsCurrent"]["units"]["USD"])
curr_assets_df

,end,val,accn,fy,fp,form,filed,frame
0,2010-12-31,235886000,0001193125-11-221497,2011,Q2,10-Q,2011-08-12,NaN
1,2010-12-31,235886000,0001193125-11-308489,2011,Q3,10-Q,2011-11-14,NaN
2,2010-12-31,235886000,0001193125-12-081990,2011,FY,10-K,2012-02-27,NaN
3,2010-12-31,235886000,0001193125-12-137560,2011,FY,10-K/A,2012-03-28,CY2010Q4I
4,2011-06-30,417758000,0001193125-11-221497,2011,Q2,10-Q,2011-08-12,CY2011Q2I
...,...,...,...,...,...,...,...,...
115,2024-12-31,58360000000,0001628280-26-003952,2025,FY,10-K,2026-01-29,CY2024Q4I
116,2025-03-31,59389000000,0001628280-25-018911,2025,Q1,10-Q,2025-04-23,CY2025Q1I
117,2025-06-30,61133000000,0001628280-25-035806,2025,Q2,10-Q,2025-07-24,CY2025Q2I
118,2025-09-30,64653000000,0001628280-25-045968,2025,Q3,10-Q,2025-10-23,CY2025Q3I


In [ ]:
# get just values reported in valid frame and plot them
curr_assets_df[curr_assets_df.frame.notna()]

,end,val,accn,fy,fp,form,filed,frame
3,2010-12-31,235886000,0001193125-12-137560,2011,FY,10-K/A,2012-03-28,CY2010Q4I
4,2011-06-30,417758000,0001193125-11-221497,2011,Q2,10-Q,2011-08-12,CY2011Q2I
5,2011-09-30,412121000,0001193125-11-308489,2011,Q3,10-Q,2011-11-14,CY2011Q3I
11,2011-12-31,372838000,0001193125-13-096241,2012,FY,10-K,2013-03-07,CY2011Q4I
12,2012-03-31,358897000,0001193125-12-225825,2012,Q1,10-Q,2012-05-10,CY2012Q1I
13,2012-06-30,317126000,0001193125-12-332138,2012,Q2,10-Q,2012-08-02,CY2012Q2I
14,2012-09-30,284541000,0001193125-12-457610,2012,Q3,10-Q,2012-11-07,CY2012Q3I
19,2012-12-31,524768000,0001193125-14-069681,2013,FY,10-K,2014-02-26,CY2012Q4I
20,2013-03-31,525993000,0001193125-13-212354,2013,Q1,10-Q,2013-05-10,CY2013Q1I
21,2013-06-30,1129542000,0001193125-13-327916,2013,Q2,10-Q,2013-08-09,CY2013Q2I


In [ ]:
import plotly.express as px
pd.options.plotting.backend = "plotly" 

curr_assets_df.plot(x="end", y="val", 
                    title=f"{company_filings['name']}, {ticker}: Current Assets",
                   labels= {
                       "val": "Value ($)",
                       "end": "Quarter End"
                   })

 ## Getting datapoints of single concept
 ## data.sec.gov/api/xbrl/companyconcept/

 The company-concept API returns all the XBRL disclosures from a single company (CIK) and concept (a taxonomy and tag) into a single JSON file, with a separate array of facts for each units on measure that the company has chosen to disclose (e.g. net profits reported in U.S. dollars and in Canadian dollars).

 * https://data.sec.gov/api/xbrl/companyconcept/CIK##########/us-gaap/AccountsPayableCurrent.json


In [ ]:
# let's retrieve current assets for comparision with company facts API
url = f"https://data.sec.gov/api/xbrl/companyconcept/CIK{str(CIK).zfill(10)}/us-gaap/AssetsCurrent.json"
url

'https://data.sec.gov/api/xbrl/companyconcept/CIK0001318605/us-gaap/AssetsCurrent.json'

In [ ]:
curr_assets_dict = requests.get(url, headers=header).json()
curr_assets_dict.keys()

dict_keys(['cik', 'taxonomy', 'tag', 'label', 'description', 'entityName', 'units'])

In [ ]:
curr_assets_dict["tag"]

'AssetsCurrent'

In [ ]:
# first 5 datapoints
curr_assets_dict["units"]["USD"][:5]

[{'end': '2010-12-31',
  'val': 235886000,
  'accn': '0001193125-11-221497',
  'fy': 2011,
  'fp': 'Q2',
  'form': '10-Q',
  'filed': '2011-08-12'},
 {'end': '2010-12-31',
  'val': 235886000,
  'accn': '0001193125-11-308489',
  'fy': 2011,
  'fp': 'Q3',
  'form': '10-Q',
  'filed': '2011-11-14'},
 {'end': '2010-12-31',
  'val': 235886000,
  'accn': '0001193125-12-081990',
  'fy': 2011,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2012-02-27'},
 {'end': '2010-12-31',
  'val': 235886000,
  'accn': '0001193125-12-137560',
  'fy': 2011,
  'fp': 'FY',
  'form': '10-K/A',
  'filed': '2012-03-28',
  'frame': 'CY2010Q4I'},
 {'end': '2011-06-30',
  'val': 417758000,
  'accn': '0001193125-11-221497',
  'fy': 2011,
  'fp': 'Q2',
  'form': '10-Q',
  'filed': '2011-08-12',
  'frame': 'CY2011Q2I'}]

In [ ]:
# this should be resulting in same DataFrame as retrieved through companyfacts API and selected through taxonomy us-gaap, AssetsCurrent concept/tag and units USD
curr_assets_df = pd.DataFrame(curr_assets_dict["units"]["USD"])
curr_assets_df

,end,val,accn,fy,fp,form,filed,frame
0,2010-12-31,235886000,0001193125-11-221497,2011,Q2,10-Q,2011-08-12,NaN
1,2010-12-31,235886000,0001193125-11-308489,2011,Q3,10-Q,2011-11-14,NaN
2,2010-12-31,235886000,0001193125-12-081990,2011,FY,10-K,2012-02-27,NaN
3,2010-12-31,235886000,0001193125-12-137560,2011,FY,10-K/A,2012-03-28,CY2010Q4I
4,2011-06-30,417758000,0001193125-11-221497,2011,Q2,10-Q,2011-08-12,CY2011Q2I
...,...,...,...,...,...,...,...,...
115,2024-12-31,58360000000,0001628280-26-003952,2025,FY,10-K,2026-01-29,CY2024Q4I
116,2025-03-31,59389000000,0001628280-25-018911,2025,Q1,10-Q,2025-04-23,CY2025Q1I
117,2025-06-30,61133000000,0001628280-25-035806,2025,Q2,10-Q,2025-07-24,CY2025Q2I
118,2025-09-30,64653000000,0001628280-25-045968,2025,Q3,10-Q,2025-10-23,CY2025Q3I


 ## Getting one fact from requested period/frame
 ## data.sec.gov/api/xbrl/frames/

 The xbrl/frames API aggregates one fact for **each** reporting entity that is last filed that most closely fits the calendrical period requested. This API supports for annual, quarterly and instantaneous data:

 * https://data.sec.gov/api/xbrl/frames/us-gaap/AccountsPayableCurrent/USD/CY2019Q1I.json

 Where the units of measure specified in the XBRL contains a numerator and a denominator, these are separated by “-per-” such as “USD-per-shares”. Note that the default unit in XBRL is “pure”.

 The period format is CY#### for annual data (duration 365 days +/- 30 days), CY####Q# for quarterly data (duration 91 days +/- 30 days), and CY####Q#I for instantaneous data. Because company financial calendars can start and end on any month or day and even change in length from quarter to quarter to according to the day of the week, the frame data is assembled by the dates that best align with a calendar quarter or year. Data users should be mindful different reporting start and end dates for facts contained in a frame.

In [ ]:
# Let's retrieve all data about current assets in Q4 of 2021
fact = "AssetsCurrent"
year = 2021
quarter = "Q1I"

url = f"https://data.sec.gov/api/xbrl/frames/us-gaap/{fact}/USD/CY{year}{quarter}.json"
url

'https://data.sec.gov/api/xbrl/frames/us-gaap/AssetsCurrent/USD/CY2021Q1I.json'

In [ ]:
curr_assets_dict = requests.get(url, headers=header).json()
curr_assets_dict.keys()

dict_keys(['taxonomy', 'tag', 'ccp', 'uom', 'label', 'description', 'pts', 'data'])

In [ ]:
# let's convert all data of requested period to pandas dataframe
curr_assets_df = pd.DataFrame(curr_assets_dict["data"])
curr_assets_df.sort_values("val", ascending=False)

,accn,cik,entityName,loc,end,val
3625,0001652044-21-000020,1652044,Alphabet Inc.,US-CA,2021-03-31,172137000000
705,0001564590-21-020891,789019,Microsoft Corporation,US-WA,2021-03-31,165614000000
1848,0001156375-21-000052,1156375,CME GROUP INC.,US-IL,2021-03-31,126745500000
486,0000320193-21-000056,320193,Apple Inc.,US-CA,2021-03-27,121465000000
1349,0001018724-21-000010,1018724,"AMAZON.COM, INC.",US-WA,2021-03-31,121408000000
...,...,...,...,...,...,...
2502,0001376474-21-000147,1411168,Dutch Oven Gold Group Inc.,CA-BC,2021-03-31,0
2654,0001477932-21-003546,1441082,THE HEALING COMPANY INC.,US-NV,2021-03-31,0
3793,0001640334-21-000895,1685570,"NANOVATION MICROTECH, INC.",-,2021-02-28,0
3303,0001640334-21-001342,1593204,"HUAIZHONG HEALTH GROUP, INC.",CN-,2021-04-30,0


In [ ]:
company_facts["facts"].keys()

dict_keys(['dei', 'us-gaap'])

In [ ]:
company_facts["entityName"]

'Tesla, Inc.'

In [ ]:
CIK = 320193
url = f"https://data.sec.gov/api/xbrl/companyconcept/CIK{str(CIK).zfill(10)}/dei/EntityRegistrantName.json"
url

'https://data.sec.gov/api/xbrl/companyconcept/CIK0000320193/dei/EntityRegistrantName.json'

 ## Retrieving Specific Tags: GrossProfit vs Custom Tags

 We can retrieve `GrossProfit` via the `companyconcept` API because it is a standard `us-gaap` tag.

 **Important Note:** Custom extensions like `SalesRevenueAutomotive` and `CostOfRevenuesAutomotive` that Tesla uses in its filings are **not available** through the `data.sec.gov` RESTful APIs (`companyconcept` and `companyfacts`). The SEC APIs strictly serve standardized taxonomy tags (`us-gaap`, `dei`, `ifrs-full`, etc.) and filter out company-specific custom extensions.

 To get custom tags like Automotive revenues, you have to use the bulk XBRL Parquet files (as you did previously) or parse the raw XMLs. We can, however, use this REST API method to fetch the standardized `GrossProfit`.

In [ ]:
# Let's switch back to Tesla's CIK to get GrossProfit
cik_tsla = "0001318605"
url_gp = f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik_tsla}/us-gaap/GrossProfit.json"

gp_response = requests.get(url_gp, headers=header)
if gp_response.status_code == 200:
    gp_dict = gp_response.json()
    gp_df = pd.DataFrame(gp_dict["units"]["USD"])
    
    # Filter for standard quarterly/annual frames to clean up the dataframe
    gp_df = gp_df[gp_df['frame'].notna()].copy()
    
    # Format the dates
    gp_df['end'] = pd.to_datetime(gp_df['end'])
    gp_df = gp_df.sort_values('end').reset_index(drop=True)
    
    print("Successfully fetched GrossProfit! Here are the most recent standard filings:\n")
    # Display the tail of the dataframe
    print(gp_df[['end', 'val', 'frame', 'form']].tail(10))
else:
    print(f"Failed to fetch GrossProfit data. Status: {gp_response.status_code}")

gp_df

Successfully fetched GrossProfit! Here are the most recent standard filings:

          end          val     frame  form
66 2023-09-30   4178000000  CY2023Q3  10-Q
67 2023-12-31  17660000000    CY2023  10-K
68 2024-03-31   3696000000  CY2024Q1  10-Q
69 2024-06-30   4578000000  CY2024Q2  10-Q
70 2024-09-30   4997000000  CY2024Q3  10-Q
71 2024-12-31  17450000000    CY2024  10-K
72 2025-03-31   3153000000  CY2025Q1  10-Q
73 2025-06-30   3878000000  CY2025Q2  10-Q
74 2025-09-30   5054000000  CY2025Q3  10-Q
75 2025-12-31  17094000000    CY2025  10-K


,start,end,val,accn,fy,fp,form,filed,frame
0,2009-01-01,2009-12-31,9535000,0001193125-12-137560,2011,FY,10-K/A,2012-03-28,CY2009
1,2010-01-01,2010-03-31,3852000,0001193125-12-137560,2011,FY,10-K/A,2012-03-28,CY2010Q1
2,2010-04-01,2010-06-30,6261000,0001193125-12-137560,2011,FY,10-K/A,2012-03-28,CY2010Q2
3,2010-07-01,2010-09-30,9296000,0001193125-12-137560,2011,FY,10-K/A,2012-03-28,CY2010Q3
4,2010-01-01,2010-12-31,30731000,0001193125-13-096241,2012,FY,10-K,2013-03-07,CY2010
...,...,...,...,...,...,...,...,...,...
71,2024-01-01,2024-12-31,17450000000,0001628280-26-003952,2025,FY,10-K,2026-01-29,CY2024
72,2025-01-01,2025-03-31,3153000000,0001628280-25-018911,2025,Q1,10-Q,2025-04-23,CY2025Q1
73,2025-04-01,2025-06-30,3878000000,0001628280-25-035806,2025,Q2,10-Q,2025-07-24,CY2025Q2
74,2025-07-01,2025-09-30,5054000000,0001628280-25-045968,2025,Q3,10-Q,2025-10-23,CY2025Q3
